# NASA CMAPSS FD001 — Evaluación rápida del dataset

**Dataset:** Kaggle `behrad3d/nasa-cmaps` (archive/CMaps/)
- `train_FD001.txt` — entrenamiento
- `test_FD001.txt` — test (sin RUL conocido, se une con RUL_FD001.txt)
- `RUL_FD001.txt` — RUL real de cada motor en test

**Objetivo de este notebook:** decidir si el dataset vale la pena escalar a un proyecto final. No es un pipeline de producción — es una prueba de viabilidad.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

## 1. Carga de datos

El dataset no tiene header. Columnas:
- `engine_id` — identificador del motor
- `cycle` — ciclo de operación (tiempo)
- `setting1`, `setting2`, `setting3` — ajustes operativos
- `s1`…`s21` — 21 sensores

In [ ]:
COLS = ['engine_id','cycle','setting1','setting2','setting3'] + [f's{i}' for i in range(1,22)]

train = pd.read_csv('archive/CMaps/train_FD001.txt', sep=r'\s+', header=None, names=COLS)
test  = pd.read_csv('archive/CMaps/test_FD001.txt',  sep=r'\s+', header=None, names=COLS)
rul   = pd.read_csv('archive/CMaps/RUL_FD001.txt',  sep=r'\s+', header=None, names=['RUL'])

print('train shape:', train.shape)
print('test  shape:', test.shape)
print('rul   shape:', rul.shape)
display(train.head(3))
display(train.tail(3))

## 2. EDA — ¿qué tenemos realmente?

In [ ]:
# ¿Cuántos motores y cómo se distribuyen los ciclos?
n_motores_train = train['engine_id'].nunique()
n_ciclos_train  = len(train)

ciclos_por_motor = train.groupby('engine_id')['cycle'].agg(['min','max','count'])
ciclos_por_motor['duracion'] = ciclos_por_motor['max'] - ciclos_por_motor['min'] + 1

print(f'Motores en train: {n_motores_train}')
print(f'Ciclos totales en train: {n_ciclos_train}')
print(f'Ciclos por motor — min {ciclos_por_motor["duracion"].min()}, '
      f'max {ciclos_por_motor["duracion"].max()}, '
      f'media {ciclos_por_motor["duracion"].mean():.1f}')
display(ciclos_por_motor.head())

In [ ]:
# Sensores constantes — ruido o senales sin variacion
sensor_cols = [c for c in COLS if c.startswith('s')]

constante_train = []
for c in sensor_cols:
    if train[c].nunique() == 1:
        constante_train.append(c)

print(f'Sensores constantes en train: {len(constante_train)} → {constante_train}')
constante_test = [c for c in sensor_cols if test[c].nunique() == 1]
print(f'Sensores constantes en test:  {len(constante_test)} → {constante_test}')

variabilidad = train[sensor_cols].std().sort_values()
fig, ax = plt.subplots(figsize=(6,6))
variabilidad.plot(kind='barh', ax=ax)
ax.set_title('Desviación estándar por sensor (train)')
ax.set_xlabel('std')
plt.tight_layout()
plt.show()

In [ ]:
# Correlación entre sensores — ¿redundancia?
corr = train[sensor_cols].corr()

fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', vmin=-1, vmax=1, ax=ax, square=True,
            cbar_kws={'shrink':0.8})
ax.set_title('Correlación entre sensores (train)')
plt.tight_layout()
plt.show()

# pares muy correlacionados
pairs = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        v = corr.iloc[i,j]
        if abs(v) > 0.9:
            pairs.append((corr.columns[i], corr.columns[j], round(v,2)))

print(f'Pares |r|>0.9: {len(pairs)}')
for p in pairs[:12]:
    print(f'  {p[0]} — {p[1]}: r={p[2]}')

In [ ]:
# ¿El RUL baja de forma tangible? Ejemplo 3 motores
train_sample = train[train['engine_id'].isin([1,2,3])]

max_cycle = train.groupby('engine_id')['cycle'].transform('max')
train_sample = train_sample.copy()
train_sample['RUL'] = max_cycle - train_sample['cycle']

fig, ax = plt.subplots(figsize=(8,4))
for eid in [1,2,3]:
    sub = train_sample[train_sample['engine_id']==eid]
    ax.plot(sub['cycle'], sub['RUL'], marker='o', label=f'Motor {eid}')
ax.set_xlabel('ciclo'); ax.set_ylabel('RUL (ciclos restantes)')
ax.set_title('RUL decreciendo — 3 motores de ejemplo')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Preprocesamiento

Reglas:
- `StandardScaler` **fit solo con train** (sin leakage).
- RUL = `max_cycle - cycle`.
- **Clip a 125**: operaciones con RUL >125 se consideran "sanas" y se valora igual (125).

In [ ]:
CLIP = 125

def prepare(df, scaler=None, fit=False):
    df = df.copy()
    max_cycle = df.groupby('engine_id')['cycle'].transform('max')
    df['RUL'] = (max_cycle - df['cycle']).clip(upper=CLIP)
    sensor_cols = [c for c in COLS if c.startswith('s')]
    X = df[sensor_cols].values
    if fit:
        scaler = scaler.fit(X)
    X_scaled = scaler.transform(X)
    y = df['RUL'].values
    out = df[['engine_id','cycle']].copy()
    out['RUL'] = y
    return X_scaled, y, out, scaler

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train, y_train, meta_train, scaler = prepare(train, scaler=scaler, fit=True)
X_test,  y_test,  meta_test,  _       = prepare(test,  scaler=scaler, fit=False)

print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_test :', X_test.shape,  'y_test :', y_test.shape)
print('RUL train — min %.0f max %.0f mean %.1f' % (y_train.min(), y_train.max(), y_train.mean()))
print('RUL test  — min %.0f max %.0f mean %.1f' % (y_test.min(),  y_test.max(),  y_test.mean()))
print('Sensor cols used:', len([c for c in COLS if c.startswith('s')]))

## 4. Ventanas para LSTM

Cada muestra = secuencia de los últimos 30 ciclos del motor → `X.shape = (n_muestras, 30, n_sensores)`.
`y` = RUL del **último ciclo** de la ventana.

In [ ]:
WINDOW = 30

def build_windows(X, y, meta, window=WINDOW):
    """
    Para cada motor: generar ventanas deslizantes de longitud `window`.
    Solo ventanas completas. y = RUL del último ciclo.
    """
    X_list, y_list = [], []
    # implementacion limpia:
    X_list, y_list = [], []
    df_meta = meta.reset_index(drop=True)
    X = np.asarray(X)
    y = np.asarray(y)
    
    unique_ids = np.unique(df_meta['engine_id'])
    for eid in unique_ids:
        idx = np.where(df_meta['engine_id'].values == eid)[0]
        seg_X = X[idx]
        seg_y = y[idx]
        if len(seg_X) < window:
            continue
        for i in range(len(seg_X) - window + 1):
            X_list.append(seg_X[i:i+window])
            y_list.append(seg_y[i+window-1])
    
    return np.array(X_list), np.array(y_list)

X_train_w, y_train_w = build_windows(X_train, y_train, meta_train, window=WINDOW)
X_test_w,  y_test_w  = build_windows(X_test,  y_test,  meta_test,  window=WINDOW)

print('Ventanas train:', X_train_w.shape, '-> y:', y_train_w.shape)
print('Ventanas test :', X_test_w.shape,  '-> y:', y_test_w.shape)
print('Ejemplo X[0].shape:', X_train_w[0].shape)
print('RUL window train mean %.1f' % y_train_w.mean())

## 5. Baseline ML — XGBoostRegressor

Features = cada ciclo individual escalado (no ventana). Evaluamos RMSE + MAE en test.

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train)

pred_test_xgb = xgb.predict(X_test)
pred_train_xgb = xgb.predict(X_train)

rmse_test_xgb  = np.sqrt(mean_squared_error(y_test, pred_test_xgb))
mae_test_xgb   = mean_absolute_error(y_test, pred_test_xgb)
rmse_train_xgb = np.sqrt(mean_squared_error(y_train, pred_train_xgb))

print(f'XGBoost  RMSE train: {rmse_train_xgb:.2f}   test: {rmse_test_xgb:.2f}')
print(f'XGBoost  MAE  test : {mae_test_xgb:.2f}')
print(f'Delta train-test   : {rmse_test_xgb - rmse_train_xgb:.2f}')

import joblib
joblib.dump(xgb, 'models/xgb_FD001_baseline.joblib')
print('Modelo guardado en models/xgb_FD001_baseline.joblib')

In [ ]:
# Plot RUL real vs predicho — 1 motor de test (el motor con más ciclos)
eid_plot = meta_test['engine_id'].value_counts().idxmax()
mask = meta_test['engine_id'] == eid_plot

cycles = meta_test.loc[mask, 'cycle'].values
rul_real = y_test[mask]
rul_pred = pred_test_xgb[mask]

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(cycles, rul_real, 'o-', label='RUL real', linewidth=2)
ax.plot(cycles, rul_pred, 's--', label='XGBoost pred', linewidth=1.5, alpha=0.8)
ax.set_xlabel('ciclo'); ax.set_ylabel('RUL')
ax.set_title(f'Motor {eid_plot} — RUL real vs prediccion (XGBoost)')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. RNN — LSTM 2 capas (100 → 50)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, callbacks, models

tf.random.set_seed(42)
np.random.seed(42)

N_SENSORS = X_train_w.shape[2]
BATCH = 64

def build_lstm():
    inp = layers.Input(shape=(WINDOW, N_SENSORS))
    x = layers.LSTM(100, return_sequences=True)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.LSTM(50)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1)(x)
    m = models.Model(inp, out, name='FD001_LSTM')
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
        loss='mse',
        metrics=['mae'],
    )
    return m

model_lstm = build_lstm()
model_lstm.summary()

In [ ]:
es = callbacks.EarlyStopping(
    monitor='val_loss', patience=12, restore_best_weights=True, verbose=1
)
rl = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

history = model_lstm.fit(
    X_train_w, y_train_w,
    validation_split=0.1,
    epochs=100,
    batch_size=BATCH,
    shuffle=True,
    callbacks=[es, rl],
    verbose=1,
)

In [ ]:
# Metricas LSTM
pred_train_lstm = model_lstm.predict(X_train_w, verbose=0).ravel()
pred_test_lstm  = model_lstm.predict(X_test_w,  verbose=0).ravel()

rmse_train_lstm = np.sqrt(mean_squared_error(y_train_w, pred_train_lstm))
rmse_test_lstm  = np.sqrt(mean_squared_error(y_test_w,  pred_test_lstm))
mae_test_lstm   = mean_absolute_error(y_test_w, pred_test_lstm)

print(f'LSTM  RMSE train: {rmse_train_lstm:.2f}   test: {rmse_test_lstm:.2f}')
print(f'LSTM  MAE  test : {mae_test_lstm:.2f}')
print(f'Delta train-test : {rmse_test_lstm - rmse_train_lstm:.2f}')

model_lstm.save('models/lstm_FD001_2layer.h5')
print('Modelo guardado en models/lstm_FD001_2layer.h5')

In [ ]:
# Curva de aprendizaje
hist = history.history
fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(hist['loss'], label='train loss')
if 'val_loss' in hist:
    ax.plot(hist['val_loss'], label='val loss')
ax.set_yscale('log')
ax.set_xlabel('epoch'); ax.set_ylabel('MSE (log)')
ax.set_title('Curva de entrenamiento LSTM')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot RUL real vs predicho con LSTM — mismo motor
eid_plot = meta_test['engine_id'].value_counts().idxmax()

test_meta_named = meta_test.reset_index(drop=True)
X_test_w_named, y_test_w_named = build_windows(X_test, y_test, test_meta_named)
pred_test_named = model_lstm.predict(X_test_w_named, verbose=0).ravel()

# Mapear predicción de ventanas al motor del último ciclo
window_to_last_cycle = []
for i, (eid, cyc) in enumerate(zip(test_meta_named['engine_id'], test_meta_named['cycle'])):
    pass

# Reconstruir mejor: predicciones por motor directamente
rows = []
unique_ids = np.unique(test_meta_named['engine_id'].values)
idx = 0
for eid in unique_ids:
    mask_e = test_meta_named['engine_id'].values == eid
    seg_len = mask_e.sum()
    if seg_len < WINDOW:
        continue
    n_win = seg_len - WINDOW + 1
    for w in range(n_win):
        rows.append({
            'engine_id': eid,
            'last_cycle': test_meta_named.loc[mask_e, 'cycle'].values[WINDOW-1+w],
            'pred': pred_test_named[idx],
        })
        idx += 1
df_pred = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8,4))
sub = df_pred[df_pred['engine_id']==eid_plot]
sub2 = test_meta_named[test_meta_named['engine_id']==eid_plot]
ax.plot(sub2['cycle'], y_test[sub2.index], 'o-', label='RUL real', linewidth=2)
ax.plot(sub['last_cycle'], sub['pred'], 's--', label='LSTM pred', linewidth=1.5, alpha=0.8)
ax.set_xlabel('ciclo'); ax.set_ylabel('RUL')
ax.set_title(f'Motor {eid_plot} — RUL real vs prediccion (LSTM)')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. V2 — comparativa ampliada y mejora LSTM

A partir de aquí todo se entrena y evalúa con **el mismo protocolo**:

- **Entrenamiento:** todas las ventanas completas de `train`, con RUL por ciclo (clip 125).
- **Test:** solo la **última ventana** de cada motor, comparada contra `RUL_FD001.txt`.
- **Train RMSE** se mide por ventana; **Test RMSE/MAE** por motor. No son la misma unidad,
  así que se reportan en columnas separadas.


In [ ]:
import json
import pathlib
import time

import joblib
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

# Mismo set de features que el baseline: 21 sensores + 3 settings = 24
SENSOR_COLS = [c for c in COLS if c.startswith('s')]
N_SENSORS = len(SENSOR_COLS)
print(f'features: {N_SENSORS} -> {SENSOR_COLS}')

# RUL real de test por motor: fuente de verdad para evaluar
rul_txt = pathlib.Path('archive/CMaps/RUL_FD001.txt').read_text(encoding='utf-8')
rul_vals = [int(x) for x in rul_txt.replace('\r', '').split()]
rul_df = pd.DataFrame({'engine_id': range(1, len(rul_vals) + 1), 'RUL_real': rul_vals})
rul_df['RUL_real'] = rul_df['RUL_real'].clip(upper=CLIP)

print(f'motores test: {len(rul_df)} | RUL real min {rul_df.RUL_real.min()} '
      f'media {rul_df.RUL_real.mean():.1f} max {rul_df.RUL_real.max()}')
display(rul_df.head(3))


### 7b. Utilidades (ventanas, métricas, guardado)

In [ ]:
def build_windows(X, y, meta, window=30):
    # Ventanas completas por motor.
    # Devuelve (Xw, yw, meta_w) con meta_w = engine_id, ciclo_fin, es_ultima.
    # `es_ultima` marca la última ventana de cada motor (la que se usa en test).
    X = np.asarray(X)
    y = np.asarray(y)
    meta = meta.reset_index(drop=True)
    cyc = meta['cycle'].values

    X_list, y_list, rows = [], [], []
    for eid in np.unique(meta['engine_id'].values):
        idx = np.where(meta['engine_id'].values == eid)[0]
        seg_X, seg_y, seg_c = X[idx], y[idx], cyc[idx]
        if len(seg_X) < window:
            continue
        for i in range(len(seg_X) - window + 1):
            X_list.append(seg_X[i:i + window])
            y_list.append(seg_y[i + window - 1])
            rows.append({
                'engine_id': int(eid),
                'ciclo_fin': int(seg_c[i + window - 1]),
                'es_ultima': bool(i + window == len(seg_X)),
            })
    return np.array(X_list), np.array(y_list), pd.DataFrame(rows)


def evaluar(y_real, y_pred, label):
    rmse = float(np.sqrt(mean_squared_error(y_real, y_pred)))
    mae = float(mean_absolute_error(y_real, y_pred))
    print(f'{label:34s} RMSE={rmse:6.2f}  MAE={mae:6.2f}')
    return rmse, mae


def evaluar_ultima(pred, meta_last):
    # Evalúa predicciones de la última ventana contra RUL_FD001.txt.
    d = meta_last.copy().reset_index(drop=True)
    d['pred'] = np.asarray(pred).ravel()
    d = d.dropna(subset=['RUL_real'])
    rmse = float(np.sqrt(mean_squared_error(d['RUL_real'], d['pred'])))
    mae = float(mean_absolute_error(d['RUL_real'], d['pred']))
    return rmse, mae, int(len(d)), d


def guardar_modelo(obj, path):
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    if path.endswith('.h5'):
        obj.save(path)
    else:
        joblib.dump(obj, path)
    print('guardado:', path)


print('helpers v2 listos')


In [ ]:
# Ventanas de 30 ciclos: todas para entrenar, última por motor para evaluar
Xtr30, ytr30, mtr30 = build_windows(X_train, y_train, meta_train, WINDOW)
Xte30_all, yte30_all, mte30_all = build_windows(X_test, y_test, meta_test, WINDOW)

mte30_last = mte30_all[mte30_all['es_ultima']].reset_index(drop=True)
mte30_last = mte30_last.merge(rul_df, on='engine_id', how='left')
Xte30_last = Xte30_all[mte30_all['es_ultima'].values]

print('ventanas train:', Xtr30.shape, '| motores test evaluados:', len(mte30_last))

# --- baseline XGBoost: predicción por ciclo promediada en la ventana ---
xgb = joblib.load('models/xgb_FD001_baseline.joblib')
pred_xgb_ciclo_te = xgb.predict(Xte30_last.reshape(-1, N_SENSORS))
pred_xgb_last = pred_xgb_ciclo_te.reshape(len(Xte30_last), WINDOW).mean(axis=1)

# --- baseline LSTM 2 capas ---
lstm2 = tf.keras.models.load_model('models/lstm_FD001_2layer.h5', compile=False)
lstm2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
              loss='mse', metrics=['mae'])
pred_lstm2_last = lstm2.predict(Xte30_last, verbose=0).ravel()

rmse_xgb, mae_xgb, n_xgb, d_xgb = evaluar_ultima(pred_xgb_last, mte30_last)
rmse_lstm2, mae_lstm2, n_lstm2, d_lstm2 = evaluar_ultima(pred_lstm2_last, mte30_last)
rmse_train_xgb = float(np.sqrt(mean_squared_error(y_train, xgb.predict(X_train))))

print(f'XGBoost  (train por ciclo {rmse_train_xgb:.2f}) -> test {rmse_xgb:.2f} / MAE {mae_xgb:.2f}')
print(f'LSTM 2c  (train ventana {rmse_train_lstm:.2f}) -> test {rmse_lstm2:.2f} / MAE {mae_lstm2:.2f}')


### 7c. Comparativa vs otros modelos (Random Forest, LightGBM, Linear Regression)

Los tres consumen la **ventana completa aplanada** (30 × 24 = 720 features) y se entrenan
con **todas** las ventanas de train. Evaluación sobre la última ventana de cada motor.

> Nota: entrenar solo con la última ventana de cada motor daría `y = 0` en todas las filas
> (la última ventana de train es, por definición, el ciclo de fallo), así que el modelo
> aprendería a predecir 0. Por eso se entrena con todas las ventanas.


In [ ]:
Xtr_flat = Xtr30.reshape(len(Xtr30), -1)
Xte_flat = Xte30_last.reshape(len(Xte30_last), -1)

res = []
t0 = time.time()
rf = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(Xtr_flat, ytr30)
t_rf = time.time() - t0
rmse_rf_tr = float(np.sqrt(mean_squared_error(ytr30, rf.predict(Xtr_flat))))
pred_rf_te = rf.predict(Xte_flat)
rmse_rf_te, mae_rf_te, _, d_rf = evaluar_ultima(pred_rf_te, mte30_last)
res.append(dict(modelo='Random Forest', rmse_tr=rmse_rf_tr, rmse_te=rmse_rf_te,
                mae_te=mae_rf_te, t=t_rf, n=len(d_rf)))
guardar_modelo(rf, 'models/rf_FD001_baseline.joblib')

try:
    import lightgbm as lgb
    t0 = time.time()
    lgb_m = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=8,
                              random_state=42, n_jobs=-1, verbose=-1)
    lgb_m.fit(Xtr_flat, ytr30)
    t_lgb = time.time() - t0
    rmse_lgb_tr = float(np.sqrt(mean_squared_error(ytr30, lgb_m.predict(Xtr_flat))))
    pred_lgb_te = lgb_m.predict(Xte_flat)
    rmse_lgb_te, mae_lgb_te, _, d_lgb = evaluar_ultima(pred_lgb_te, mte30_last)
    res.append(dict(modelo='LightGBM', rmse_tr=rmse_lgb_tr, rmse_te=rmse_lgb_te,
                    mae_te=mae_lgb_te, t=t_lgb, n=len(d_lgb)))
    guardar_modelo(lgb_m, 'models/lgb_FD001_baseline.joblib')
except Exception as exc:  # lightgbm no instalado
    print('LightGBM no disponible:', exc)
    pred_lgb_te = np.full(len(mte30_last), np.nan)
    rmse_lgb_te = mae_lgb_te = float('nan')
    res.append(dict(modelo='LightGBM', rmse_tr=float('nan'), rmse_te=float('nan'),
                    mae_te=float('nan'), t=float('nan'), n=0))

t0 = time.time()
lr_pipe = make_pipeline(StandardScaler(), LinearRegression())
lr_pipe.fit(Xtr_flat, ytr30)
t_lr = time.time() - t0
rmse_lr_tr = float(np.sqrt(mean_squared_error(ytr30, lr_pipe.predict(Xtr_flat))))
pred_lr_te = lr_pipe.predict(Xte_flat)
rmse_lr_te, mae_lr_te, _, d_lr = evaluar_ultima(pred_lr_te, mte30_last)
res.append(dict(modelo='Linear Regression', rmse_tr=rmse_lr_tr, rmse_te=rmse_lr_te,
                mae_te=mae_lr_te, t=t_lr, n=len(d_lr)))
guardar_modelo(lr_pipe, 'models/lr_FD001_baseline.joblib')

tabla = pd.DataFrame(res)[['modelo', 'rmse_tr', 'rmse_te', 'mae_te', 't', 'n']]
tabla.columns = ['Modelo', 'RMSE train (ventana)', 'RMSE test (motor)', 'MAE test', 'Tiempo (s)', 'n']
print('\n=== RF / LightGBM / Linear Regression ===')
display(tabla.round(2))


### 7d. Mejora LSTM

Tres variantes sobre la misma base, entrenadas con todas las ventanas:

1. **A** — LSTM 3 capas 100 → 64 → 32 con Dropout 0.2 (w=30).
2. **B** — Bidirectional LSTM 100 → 50 con Dropout 0.2 (w=30).
3. **C** — LSTM 2 capas 100 → 50 con ventana de 50 ciclos.

La ventana de 50 solo existe para motores con ≥ 50 ciclos, así que la selección se hace
**a igual número de motores** (intersección con el subconjunto de w=50). El ganador se
guarda en `models/lstm_FD001_best.h5`.


In [ ]:
def build_lstm_multi(inp_shape, units=(100, 64, 32), bidirectional=False, dropout=0.2):
    # Construye exactamente len(units) capas LSTM (la última sin return_sequences).
    inp = layers.Input(shape=inp_shape)
    x = inp
    for i, u in enumerate(units):
        last = (i == len(units) - 1)
        if bidirectional:
            x = layers.Bidirectional(layers.LSTM(u, return_sequences=not last))(x)
        else:
            x = layers.LSTM(u, return_sequences=not last)(x)
        x = layers.BatchNormalization()(x)
        if dropout and not last:
            x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu')(x)
    if dropout:
        x = layers.Dropout(dropout)(x)
    out = layers.Dense(1)(x)
    m = models.Model(inp, out, name='LSTM_variante')
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
              loss='mse', metrics=['mae'])
    return m


def entrenar(model, X, y, epochs=60, batch=64, patience=8):
    es = callbacks.EarlyStopping(monitor='val_loss', patience=patience,
                                 restore_best_weights=True, verbose=1)
    rl = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                                     min_lr=1e-6, verbose=1)
    t0 = time.time()
    h = model.fit(X, y, validation_split=0.1, epochs=epochs, batch_size=batch,
                  shuffle=True, callbacks=[es, rl], verbose=2)
    return h, time.time() - t0


W50 = 50
Xtr50, ytr50, mtr50 = build_windows(X_train, y_train, meta_train, W50)
Xte50_all, yte50_all, mte50_all = build_windows(X_test, y_test, meta_test, W50)

mte50_last = mte50_all[mte50_all['es_ultima']].reset_index(drop=True)
mte50_last = mte50_last.merge(rul_df, on='engine_id', how='left')
Xte50_last = Xte50_all[mte50_all['es_ultima'].values]
print('motores con ventana de 50:', len(mte50_last))

specs = [
    ('A: LSTM 3 capas 100-64-32 + Dropout 0.2 (w=30)',
     dict(units=(100, 64, 32), bidirectional=False, dropout=0.2), WINDOW),
    ('B: Bidirectional LSTM 100-50 + Dropout 0.2 (w=30)',
     dict(units=(100, 50), bidirectional=True, dropout=0.2), WINDOW),
    ('C: LSTM 2 capas 100-50 + Dropout 0.2 (w=50)',
     dict(units=(100, 50), bidirectional=False, dropout=0.2), W50),
]

variantes = []
for nombre, kw, w in specs:
    if w == WINDOW:
        Xtr_w, ytr_w, Xte_w, mte_w = Xtr30, ytr30, Xte30_last, mte30_last
    else:
        Xtr_w, ytr_w, Xte_w, mte_w = Xtr50, ytr50, Xte50_last, mte50_last

    print('\n===', nombre, '===')
    m = build_lstm_multi((w, N_SENSORS), **kw)
    hist, ttime = entrenar(m, Xtr_w, ytr_w)
    pred_tr = m.predict(Xtr_w, verbose=0).ravel()
    pred_te = m.predict(Xte_w, verbose=0).ravel()
    rmse_tr = float(np.sqrt(mean_squared_error(ytr_w, pred_tr)))
    rmse_te, mae_te, n, d = evaluar_ultima(pred_te, mte_w)
    print(f'{nombre}: RMSE train {rmse_tr:.2f} | RMSE test {rmse_te:.2f} | '
          f'MAE {mae_te:.2f} | n={n} | {ttime:.1f}s')
    variantes.append(dict(nombre=nombre, modelo=m, hist=hist, w=w, rmse_tr=rmse_tr,
                          rmse_te=rmse_te, mae_te=mae_te, n=n, d=d, t=ttime))

# comparación justa: mismos motores para las tres variantes
ids50 = set(mte50_last['engine_id'].tolist())
for v in variantes:
    sub = v['d'][v['d']['engine_id'].isin(ids50)]
    v['rmse_te_comun'] = float(np.sqrt(mean_squared_error(sub['RUL_real'], sub['pred'])))
    v['n_comun'] = int(len(sub))

best = min(variantes, key=lambda v: v['rmse_te_comun'])
best_model, BEST_W = best['modelo'], best['w']

print('\n=== MEJOR LSTM (a igual n = %d motores) ===' % best['n_comun'])
print(f"{best['nombre']} -> RMSE test {best['rmse_te_comun']:.2f} "
      f"(todos sus motores: {best['rmse_te']:.2f}, n={best['n']})")

guardar_modelo(best_model, 'models/lstm_FD001_best.h5')
pathlib.Path('models/lstm_FD001_best.json').write_text(
    json.dumps({'nombre': best['nombre'], 'window': BEST_W,
                'rmse_test': best['rmse_te'], 'mae_test': best['mae_te']},
               ensure_ascii=False, indent=1), encoding='utf-8')

var_df = pd.DataFrame([{
    'Variante': v['nombre'],
    'RMSE train': round(v['rmse_tr'], 2),
    'RMSE test': round(v['rmse_te'], 2),
    'RMSE test (mismo n)': round(v['rmse_te_comun'], 2),
    'MAE test': round(v['mae_te'], 2),
    'n motores': v['n'],
    'Tiempo (s)': round(v['t'], 1),
} for v in variantes])
print('\n=== VARIANTES LSTM ===')
display(var_df)


In [ ]:
# Tabla única de todos los modelos del notebook
filas = [
    dict(Modelo='XGBoost (baseline, por ciclo)', rmse_tr=rmse_train_xgb, rmse_te=rmse_xgb,
         mae_te=mae_xgb, t=np.nan, n=n_xgb),
    dict(Modelo='Random Forest (ventana aplanada)', rmse_tr=res[0]['rmse_tr'],
         rmse_te=res[0]['rmse_te'], mae_te=res[0]['mae_te'], t=res[0]['t'], n=res[0]['n']),
    dict(Modelo='LightGBM (ventana aplanada)', rmse_tr=res[1]['rmse_tr'],
         rmse_te=res[1]['rmse_te'], mae_te=res[1]['mae_te'], t=res[1]['t'], n=res[1]['n']),
    dict(Modelo='Linear Regression (ventana aplanada)', rmse_tr=res[2]['rmse_tr'],
         rmse_te=res[2]['rmse_te'], mae_te=res[2]['mae_te'], t=res[2]['t'], n=res[2]['n']),
    dict(Modelo='LSTM 2 capas 100-50 (baseline)', rmse_tr=rmse_train_lstm,
         rmse_te=rmse_lstm2, mae_te=mae_lstm2, t=np.nan, n=n_lstm2),
    dict(Modelo=f"LSTM best: {best['nombre']}", rmse_tr=best['rmse_tr'],
         rmse_te=best['rmse_te'], mae_te=best['mae_te'], t=best['t'], n=best['n']),
]
tabla_final = pd.DataFrame(filas)[['Modelo', 'rmse_tr', 'rmse_te', 'mae_te', 't', 'n']]
tabla_final.columns = ['Modelo', 'RMSE train', 'RMSE test', 'MAE test', 'Tiempo (s)', 'n motores']
print('=== TABLA COMPARATIVA (test: última ventana vs RUL_FD001.txt, clip 125) ===')
display(tabla_final.round(2))


### 7e. Gráficos obligatorios

1. Cuatro motores aleatorios: RUL real vs predicción XGBoost vs predicción del mejor LSTM, **por ciclo**.
   El RUL real por ciclo se reconstruye como `RUL_FD001 + (ciclo_max - ciclo)`, que es exactamente
   el RUL verdadero de cada ciclo del motor.
2. Scatter RUL real vs predicho del mejor LSTM (última ventana por motor) con diagonal ideal.
3. Histograma de residuos del mejor LSTM.
4. Curva de aprendizaje (loss train/val) del mejor LSTM.


In [ ]:
# RUL real por ciclo en test (reconstruido desde RUL_FD001.txt)
real_ciclo = meta_test.copy()
real_ciclo['ciclo_max'] = real_ciclo.groupby('engine_id')['cycle'].transform('max')
real_ciclo = real_ciclo.merge(rul_df[['engine_id', 'RUL_real']], on='engine_id', how='left')
real_ciclo['RUL_ciclo'] = real_ciclo['RUL_real'] + (real_ciclo['ciclo_max'] - real_ciclo['cycle'])
real_ciclo['pred_xgb'] = xgb.predict(X_test)

# Predicciones por ciclo del mejor LSTM (ventana BEST_W)
Xte_best_all, _, mte_best_all = build_windows(X_test, y_test, meta_test, BEST_W)
mte_best_all['pred_lstm'] = best_model.predict(Xte_best_all, verbose=0).ravel()
serie = real_ciclo.merge(
    mte_best_all[['engine_id', 'ciclo_fin', 'pred_lstm']],
    left_on=['engine_id', 'cycle'], right_on=['engine_id', 'ciclo_fin'], how='left')

candidatos = sorted(serie.loc[serie['pred_lstm'].notna(), 'engine_id'].unique())
np.random.seed(7)
motores = sorted(np.random.choice(candidatos, size=min(4, len(candidatos)), replace=False).tolist())
print('motores representados:', motores)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, eid in zip(axes.ravel(), motores):
    sub = serie[serie['engine_id'] == eid].sort_values('cycle')
    ax.plot(sub['cycle'], sub['RUL_ciclo'], 'k-', lw=2.2, label='RUL real')
    ax.plot(sub['cycle'], sub['pred_xgb'], '--', color='steelblue', lw=1.5, label='XGBoost')
    ax.plot(sub['cycle'], sub['pred_lstm'], '-', color='crimson', lw=1.5,
            label=f'LSTM ({BEST_W} ciclos)')
    ax.axhline(15, color='orange', ls=':', lw=1)
    ax.axhline(30, color='green', ls=':', lw=1)
    ax.set_title(f'Motor {eid}')
    ax.set_xlabel('ciclo')
    ax.set_ylabel('RUL (clip 125)')
    ax.invert_yaxis()
    ax.legend(fontsize=8)
fig.suptitle('RUL real vs predicción XGBoost vs mejor LSTM — 4 motores de test')
plt.tight_layout()
plt.show()


In [ ]:
# Scatter y residuos del mejor LSTM sobre la última ventana por motor
if BEST_W == WINDOW:
    Xbest_last, mbest_last = Xte30_last, mte30_last
else:
    Xbest_last, mbest_last = Xte50_last, mte50_last

pred_best_last = best_model.predict(Xbest_last, verbose=0).ravel()
rmse_best, mae_best, n_best, d_best = evaluar_ultima(pred_best_last, mbest_last)
print(f"mejor LSTM -> RMSE {rmse_best:.2f} / MAE {mae_best:.2f} (n={n_best})")

real = d_best['RUL_real'].values
pred = d_best['pred'].values
resid = real - pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(real, pred, alpha=0.75, s=26, edgecolor='k', linewidth=0.3)
lim = [0, max(real.max(), pred.max()) + 5]
axes[0].plot(lim, lim, 'r--', lw=1.5, label='diagonal ideal')
axes[0].set_xlim(lim)
axes[0].set_ylim(lim)
axes[0].set_xlabel('RUL real (clip 125)')
axes[0].set_ylabel('RUL predicha')
axes[0].set_title(f'Scatter — mejor LSTM (RMSE {rmse_best:.2f})')
axes[0].legend()

axes[1].hist(resid, bins=20, color='steelblue', edgecolor='black')
axes[1].axvline(0, color='red', ls='--', lw=1.5)
axes[1].set_xlabel('residuo (RUL real - RUL predicha)')
axes[1].set_ylabel('frecuencia')
axes[1].set_title(f'Residuos — media {resid.mean():.2f}, std {resid.std():.2f}')
plt.tight_layout()
plt.show()


In [ ]:
# Curva de aprendizaje del mejor LSTM
h = best['hist'].history
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(h['loss'], label='train loss')
if 'val_loss' in h:
    ax.plot(h['val_loss'], label='val loss')
ax.set_yscale('log')
ax.set_xlabel('epoch')
ax.set_ylabel('MSE (log)')
ax.set_title(f"Curva de entrenamiento — {best['nombre']}")
ax.legend()
plt.tight_layout()
plt.show()
print(f"epochs ejecutadas: {len(h['loss'])} | mejor val_loss: {min(h.get('val_loss', [np.nan])):.2f}")


### 7f. Feature importance de XGBoost

En FD001 los sensores más informativos suelen ser `s2, s3, s4, s7, s11, s12, s15`.
Se marcan sobre la importancia real del baseline entrenado por ciclo.


In [ ]:
fi_df = pd.DataFrame({'sensor': SENSOR_COLS, 'importance': xgb.feature_importances_})
fi_df = fi_df.sort_values('importance', ascending=False).reset_index(drop=True)
key = ['s2', 's3', 's4', 's7', 's11', 's12', 's15']
fi_df['es_key'] = fi_df['sensor'].isin(key)

print('Top 10 features por importancia:')
display(fi_df.head(10)[['sensor', 'importance', 'es_key']].round(4))

plt.figure(figsize=(9, 6))
sns.barplot(data=fi_df, y='sensor', x='importance', hue='es_key', dodge=False, legend=False,
            palette={True: 'crimson', False: 'steelblue'})
plt.xlabel('importancia (gain, escala log)')
plt.ylabel('feature')
plt.xscale('log')
plt.title('Feature importance XGBoost — sensores clave en rojo')
plt.tight_layout()
plt.show()

print('sensores clave en el top 10:', [s for s in fi_df.head(10)['sensor'] if s in key])


### 7g. Motor de reglas prototipo (semáforo)

- **ROJO**: RUL < 15
- **AMARILLO**: 15 ≤ RUL < 30
- **VERDE**: RUL ≥ 30

Se aplica a las predicciones de cada modelo sobre la última ventana de cada motor y se
cuenta cuántos motores caen en cada zona. También se compara con el semáforo real.


In [ ]:
def semaforo(rul):
    if rul < 15:
        return 'ROJO'
    elif rul < 30:
        return 'AMARILLO'
    return 'VERDE'


eval_df = mte30_last[['engine_id', 'RUL_real']].copy()
eval_df['pred_xgb'] = pred_xgb_last
eval_df['pred_lstm'] = pred_lstm2_last
eval_df['pred_rf'] = pred_rf_te
eval_df['pred_lgb'] = pred_lgb_te
eval_df['pred_lr'] = pred_lr_te
eval_df['pred_lstm_best'] = eval_df['engine_id'].map(
    pd.Series(pred_best_last, index=d_best['engine_id'].values))
eval_df.to_csv('models/test_eval_ultima_ventana.csv', index=False)
print('tabla guardada en models/test_eval_ultima_ventana.csv')
display(eval_df.head(5).round(2))

modelos = [
    ('XGBoost', 'pred_xgb'),
    ('Random Forest', 'pred_rf'),
    ('LightGBM', 'pred_lgb'),
    ('Linear Regression', 'pred_lr'),
    ('LSTM 2 capas', 'pred_lstm'),
    (f'LSTM best ({BEST_W} ciclos)', 'pred_lstm_best'),
]

filas_sem = []
for nombre, col in modelos:
    sub = eval_df[['RUL_real', col]].dropna()
    c = sub[col].apply(semaforo).value_counts().reindex(['ROJO', 'AMARILLO', 'VERDE']).fillna(0).astype(int)
    real_c = sub['RUL_real'].apply(semaforo).value_counts().reindex(['ROJO', 'AMARILLO', 'VERDE']).fillna(0).astype(int)
    aciertos = int((sub[col].apply(semaforo).values == sub['RUL_real'].apply(semaforo).values).sum())
    filas_sem.append({'Modelo': nombre, 'n': len(sub), 'ROJO': c['ROJO'], 'AMARILLO': c['AMARILLO'],
                      'VERDE': c['VERDE'], 'aciertos zona': aciertos,
                      'aciertos zona (%)': round(100 * aciertos / len(sub), 1)})

sem_df = pd.DataFrame(filas_sem)
real_sem = eval_df['RUL_real'].apply(semaforo).value_counts().reindex(['ROJO', 'AMARILLO', 'VERDE']).fillna(0).astype(int)
print('\n=== SEMÁFORO REAL (RUL_FD001.txt, n=%d) ===' % len(eval_df))
print(f"ROJO {real_sem['ROJO']} | AMARILLO {real_sem['AMARILLO']} | VERDE {real_sem['VERDE']}")
print('\n=== SEMÁFORO POR MODELO ===')
display(sem_df)


In [ ]:
# Métricas de la ejecución -> JSON (lo usa scripts/finalize_v2.py para la conclusión y el .md)
metricas = {
    'xgb': {'rmse_train': round(rmse_train_xgb, 2), 'rmse_test': round(rmse_xgb, 2),
            'mae_test': round(mae_xgb, 2), 'n': int(n_xgb)},
    'lstm2': {'rmse_train': round(rmse_train_lstm, 2), 'rmse_test': round(rmse_lstm2, 2),
              'mae_test': round(mae_lstm2, 2), 'n': int(n_lstm2)},
    'rf': {'rmse_train': round(res[0]['rmse_tr'], 2), 'rmse_test': round(res[0]['rmse_te'], 2),
           'mae_test': round(res[0]['mae_te'], 2), 't': round(res[0]['t'], 1), 'n': int(res[0]['n'])},
    'lgb': {'rmse_train': round(res[1]['rmse_tr'], 2), 'rmse_test': round(res[1]['rmse_te'], 2),
            'mae_test': round(res[1]['mae_te'], 2), 't': round(res[1]['t'], 1), 'n': int(res[1]['n'])},
    'lr': {'rmse_train': round(res[2]['rmse_tr'], 2), 'rmse_test': round(res[2]['rmse_te'], 2),
           'mae_test': round(res[2]['mae_te'], 2), 't': round(res[2]['t'], 1), 'n': int(res[2]['n'])},
    'best': {'nombre': best['nombre'], 'window': int(BEST_W),
             'rmse_train': round(best['rmse_tr'], 2), 'rmse_test': round(best['rmse_te'], 2),
             'mae_test': round(best['mae_te'], 2), 'n': int(best['n']),
             'rmse_test_mismo_n': round(best['rmse_te_comun'], 2), 'n_mismo_n': int(best['n_comun']),
             'epochs': len(best['hist'].history['loss']), 'tiempo': round(best['t'], 1)},
    'variantes': [{'nombre': v['nombre'], 'rmse_train': round(v['rmse_tr'], 2),
                   'rmse_test': round(v['rmse_te'], 2), 'mae_test': round(v['mae_te'], 2),
                   'rmse_test_mismo_n': round(v['rmse_te_comun'], 2), 'n': int(v['n']),
                   'tiempo': round(v['t'], 1)} for v in variantes],
    'semaforo_real': {k: int(v) for k, v in real_sem.items()},
    'semaforo': sem_df.rename(columns={'aciertos zona (%)': 'aciertos_pct'}).to_dict(orient='records'),
}
pathlib.Path('models/v2_metrics.json').write_text(
    json.dumps(metricas, ensure_ascii=False, indent=1), encoding='utf-8')
print('métricas guardadas en models/v2_metrics.json')


In [ ]:
import sys
sys.path.insert(0, 'scripts')

import json
import pathlib
import importlib

import v3_core
importlib.reload(v3_core)
from lightgbm import LGBMRegressor

# Demo en vivo del protocolo: OOF por motor + umbral por coste (FD001, features de salud)
tr3, te3, rul3 = v3_core.load_fd('FD001')
_, f3 = v3_core.add_health_features(tr3)
_, f3t = v3_core.add_health_features(te3)
X3, y3, m3 = v3_core.make_windows(tr3, f3)
X3t, m3t = v3_core.test_last(te3, f3t, rul3)

mk3 = lambda s: LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=8,
                              random_state=s, n_jobs=-1, verbose=-1)
oof3 = v3_core.oof_predict(X3, y3, m3.engine_id.values, mk3, folds=3, seeds=(42,))[42]
mod3 = mk3(42).fit(X3.reshape(len(X3), -1), y3)
p3 = mod3.predict(X3t.reshape(len(X3t), -1))
cal3 = v3_core.calibrate_threshold(oof3, y3)

print(f'features: {f3.shape[1]} | ventanas train: {X3.shape}')
print(f'RMSE OOF por motor: {v3_core.rmse(y3, oof3):.2f}  (in-sample era 7.34)')
print(f'RMSE test (motor):  {v3_core.rmse(m3t.RUL_real, p3):.2f}')
print('umbral calibrado en OOF:', cal3)
print('test con T*:', v3_core.eval_alarm(p3, m3t.RUL_real.values, cal3['T']))
print('test con 15 fijo:', v3_core.eval_alarm(p3, m3t.RUL_real.values, 15.0))


### 9b. Resultados de la fase 0-2 (OOF por motor + umbral por coste)

`models/v3_results.json` recoge todas las combinaciones ejecutadas. En FD001 los modelos
tabulares usan 5 folds × 3 seeds; en FD002/FD004, 3 folds × 1 seed (coste). La fila de la LSTM
usa **holdout por motor** (20 % de motores fuera), no OOF, y su columna es RMSE de validación.


In [ ]:
res = json.loads(pathlib.Path('models/v3_results.json').read_text(encoding='utf-8'))
filas = []
for fd, d in res.items():
    for clave, v in d.get('models', {}).items():
        filas.append({
            'Dataset': fd, 'Features': v['features'], 'Modelo': v['modelo'],
            'RMSE val/OOF': v.get('rmse_oof_mean', v.get('rmse_val')),
            '±std': v.get('rmse_oof_std', 0),
            'RMSE test': v['rmse_test'], 'MAE test': v['mae_test'],
            'T*': v['T'],
            'Recall @T*': v['alarma_T']['recall'], 'Prec @T*': v['alarma_T']['precision'],
            'Alarmas': v['alarma_T']['alarmas'],
            'Recall @15': v['alarma_15']['recall'],
        })
tab_v3 = pd.DataFrame(filas)
print('=== FASE 0-2: OOF por motor, test por motor, alarma calibrada ===')
display(tab_v3.sort_values(['Dataset', 'RMSE test']).reset_index(drop=True))
print('Nota: RMSE OOF se mide por ventana; RMSE test por motor. La columna ±std viene de 3 seeds en FD001.')


### 9c. Fase 3 — incertidumbre (LightGBM quantile)

Intervalo del 80 % (`q10`–`q90`) sobre la última ventana de cada motor. La regla ingenua
`q90 < 30` no dispara nunca (el intervalo es demasiado ancho); se comparan varias reglas y
se elige la de menor coste.


In [ ]:
q = pd.read_csv('models/v3_quantile_FD001.csv')
pol = json.loads(pathlib.Path('models/alarm_policy.json').read_text(encoding='utf-8'))
real_q = q['RUL_real'].values
cover = float(np.mean((real_q >= q['q10'].values) & (real_q <= q['q90'].values)))
ancho = float(np.mean(q['q90'].values - q['q10'].values))
print(f'cobertura del intervalo 80%: {100*cover:.1f}%  | ancho medio: {ancho:.1f} ciclos')

orden = q.sort_values('RUL_real').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(orden.index, orden['RUL_real'], 'k-', lw=2, label='RUL real')
ax.plot(orden.index, orden['pred'], color='steelblue', lw=1.2, label='RUL predicha')
ax.fill_between(orden.index, orden['q10'], orden['q90'], color='steelblue', alpha=0.15,
                label='intervalo 80%')
ax.axhline(pol['T_crit'], color='crimson', ls='--', lw=1.2, label=f"T crítico = {pol['T_crit']}")
ax.set_xlabel('motores ordenados por RUL real')
ax.set_ylabel('RUL (clip 125)')
ax.set_title('Fase 3 — predicción e intervalo del 80 % (última ventana por motor)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

reglas = pd.DataFrame(pol['reglas']).T.reset_index().rename(columns={'index': 'Regla'})
print('=== comparación de reglas de alarma (test, 100 motores) ===')
display(reglas[['Regla', 'alarmas', 'recall', 'precision', 'coste']].sort_values('coste'))
print('regla elegida:', pol['regla_elegida'])


### 9d. Fase 4 — transferencia a FD002 (6 condiciones) y FD004 (6 condiciones, 2 modos de fallo)

Mismo pipeline, recalibrando el umbral por coste en cada dataset. Se comparan solo los
modelos tabulares: la LSTM se reentrena únicamente en FD001 y FD002 (coste de cómputo).


In [ ]:
trans = tab_v3[tab_v3['Features'] == 'health'][
    ['Dataset', 'Modelo', 'RMSE OOF', 'RMSE test', 'MAE test', 'T*', 'Recall @T*', 'Alarmas']
].copy()
print('=== FASE 4: transferencia del pipeline ===')
display(trans.sort_values(['Dataset', 'RMSE test']).reset_index(drop=True))

piv = tab_v3[tab_v3['Features'] == 'health'].pivot_table(
    index='Modelo', columns='Dataset', values='RMSE test')
print('\nRMSE test por dataset (features de salud):')
display(piv.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
piv.T.plot(kind='bar', ax=ax)
ax.set_ylabel('RMSE test')
ax.set_title('Transferencia: RMSE test por dataset y modelo')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


### 9e. Fase 5 — sistema: inferencia, política y drift

`scripts/predict.py` carga `models/v3_bundle.joblib` (modelo + quantiles + features + versión)
y `models/alarm_policy.json`, y devuelve RUL, intervalo, alarma y **coste esperado de la decisión**.
Para drift: PSI **a nivel de flota** (comparar un solo motor contra la flota mide heterogeneidad,
no drift) y, por motor, % de valores fuera del rango [p01, p99] de entrenamiento.


In [ ]:
import predict
importlib.reload(predict)

_, te5, _ = v3_core.load_fd('FD001')
for eid in (1, 3, 20):
    g = te5[te5['engine_id'] == eid]
    r = predict.predict(g)
    print(f'motor {eid:3d} -> RUL {r["rul"]:6.1f} [{r["q10"]:5.1f}, {r["q90"]:5.1f}]  '
          f'{r["alerta"]:8s} coste_esperado={r["coste_esperado"]:.2f}')

inf, met = predict.report('FD001')
print('\ninforme batch (100 motores):', met)
display(inf.head(5))

dfleet = predict.drift_batch(te5)
print(f"\ndrift de flota -> PSI max {dfleet['psi_max']:.3f} en {dfleet['peor_sensor']} "
      f"(alerta={dfleet['alerta_drift']}, n={dfleet['n_filas']})")
o = predict.out_of_range(te5[te5['engine_id'] == 66])
print(f"motor 66 fuera de rango: {o['peor_pct']:.1f}% en {o['peor_sensor']} (alerta={o['alerta']})")


### 9f. Limitaciones de la v3 (dichas antes de que las pregunte nadie)

- **La `q90 < 30` no funciona**: el intervalo del 80 % mide ~65 ciclos de ancho, así que casi ningún
  motor queda por debajo del umbral. La incertidumbre sirve para priorizar, no para disparar alarmas sola.
- **El umbral se mueve con el dataset**: T* = 22 en FD001, 42.5 en FD002, 45 en FD004. No es transportable
  sin recalibrar (y recalibrar exige fallos reales).
- **OOF mide ventanas, test mide motores**: siguen siendo unidades distintas; el OOF es más pesimista.
- **FD002/FD004 con 1 seed y 3 folds**: sin barra de error. Sirven para ver si transfiere, no para decidir.
- **La LSTM** solo se reentrena en FD001/FD002; en FD004 no se ejecutó por coste (~20 min).
- Un **split temporal** de la RNN se probó y se descartó: validaba bien (14.5) pero en test daba 63,
  porque el tramo final de cada motor tiene un rango de RUL estrecho y el modelo no aprende a distinguir
  motores sanos. El holdout por motor sí cubre todo el rango.
- **El coste 20:1 es una suposición**, no un dato de planta. Cambiarlo mueve T* y el recall.


## 9. V3 — validación honesta, alarma por coste e incertidumbre

La v2 medía el train **in-sample** (7.34) y alertaba con umbrales fijos. La v3 corrige eso:

1. **Validación por motor**: `GroupKFold` agrupando por `engine_id`, así un motor nunca está
   en train y validación a la vez. La RNN usa **holdout por motor** (20 % de los motores fuera)
   porque 5 folds × LSTM no es viable. Se probó antes un split temporal (70/30 de ciclos) y se
   descartó: solo ve el final de la vida del motor, no aprende el rango completo de RUL y en test
   se iba a RMSE 63.
2. **Umbral calibrado por coste** (`FN × 20 + FP × 1`) sobre OOF de train, nunca sobre test.
3. **Features de salud** causales (solo pasado): z-score por motor, deltas a 10 y 30, std rolling 15,
   media 5 sobre los 7 sensores clave. Se aplican **igual** a GBDT y a LSTM.
4. **Incertidumbre** con LightGBM quantile (α = 0.1 / 0.9).
5. **Transferencia** del pipeline a FD002 y FD004.

El protocolo de test no cambia: última ventana por motor contra `RUL_FDxxx.txt` con clip 125.


In [ ]:
import sys
sys.path.insert(0, 'scripts')

import json
import pathlib
import importlib

import v3_core
importlib.reload(v3_core)
from lightgbm import LGBMRegressor

# Demo en vivo del protocolo: OOF por motor + umbral por coste (FD001, features de salud)
tr3, te3, rul3 = v3_core.load_fd('FD001')
_, f3 = v3_core.add_health_features(tr3)
_, f3t = v3_core.add_health_features(te3)
X3, y3, m3 = v3_core.make_windows(tr3, f3)
X3t, m3t = v3_core.test_last(te3, f3t, rul3)

mk3 = lambda s: LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=8,
                              random_state=s, n_jobs=-1, verbose=-1)
oof3 = v3_core.oof_predict(X3, y3, m3.engine_id.values, mk3, folds=3, seeds=(42,))[42]
mod3 = mk3(42).fit(X3.reshape(len(X3), -1), y3)
p3 = mod3.predict(X3t.reshape(len(X3t), -1))
cal3 = v3_core.calibrate_threshold(oof3, y3)

print(f'features: {f3.shape[1]} | ventanas train: {X3.shape}')
print(f'RMSE OOF por motor: {v3_core.rmse(y3, oof3):.2f}  (in-sample era 7.34)')
print(f'RMSE test (motor):  {v3_core.rmse(m3t.RUL_real, p3):.2f}')
print('umbral calibrado en OOF:', cal3)
print('test con T*:', v3_core.eval_alarm(p3, m3t.RUL_real.values, cal3['T']))
print('test con 15 fijo:', v3_core.eval_alarm(p3, m3t.RUL_real.values, 15.0))


### 9b. Resultados de la fase 0-2 (OOF por motor + umbral por coste)

`models/v3_results.json` recoge todas las combinaciones ejecutadas. En FD001 los modelos
tabulares usan 5 folds × 3 seeds; en FD002/FD004, 3 folds × 1 seed (coste). La fila de la LSTM
usa **holdout por motor** (20 % de motores fuera), no OOF, y su columna es RMSE de validación.


In [ ]:
res = json.loads(pathlib.Path('models/v3_results.json').read_text(encoding='utf-8'))
filas = []
for fd, d in res.items():
    for clave, v in d.get('models', {}).items():
        filas.append({
            'Dataset': fd, 'Features': v['features'], 'Modelo': v['modelo'],
            'RMSE val/OOF': v.get('rmse_oof_mean', v.get('rmse_val')),
            '±std': v.get('rmse_oof_std', 0),
            'RMSE test': v['rmse_test'], 'MAE test': v['mae_test'],
            'T*': v['T'],
            'Recall @T*': v['alarma_T']['recall'], 'Prec @T*': v['alarma_T']['precision'],
            'Alarmas': v['alarma_T']['alarmas'],
            'Recall @15': v['alarma_15']['recall'],
        })
tab_v3 = pd.DataFrame(filas)
print('=== FASE 0-2: OOF por motor, test por motor, alarma calibrada ===')
display(tab_v3.sort_values(['Dataset', 'RMSE test']).reset_index(drop=True))
print('Nota: RMSE OOF se mide por ventana; RMSE test por motor. La columna ±std viene de 3 seeds en FD001.')


### 9c. Fase 3 — incertidumbre (LightGBM quantile)

Intervalo del 80 % (`q10`–`q90`) sobre la última ventana de cada motor. La regla ingenua
`q90 < 30` no dispara nunca (el intervalo es demasiado ancho); se comparan varias reglas y
se elige la de menor coste.


In [ ]:
q = pd.read_csv('models/v3_quantile_FD001.csv')
pol = json.loads(pathlib.Path('models/alarm_policy.json').read_text(encoding='utf-8'))
real_q = q['RUL_real'].values
cover = float(np.mean((real_q >= q['q10'].values) & (real_q <= q['q90'].values)))
ancho = float(np.mean(q['q90'].values - q['q10'].values))
print(f'cobertura del intervalo 80%: {100*cover:.1f}%  | ancho medio: {ancho:.1f} ciclos')

orden = q.sort_values('RUL_real').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(orden.index, orden['RUL_real'], 'k-', lw=2, label='RUL real')
ax.plot(orden.index, orden['pred'], color='steelblue', lw=1.2, label='RUL predicha')
ax.fill_between(orden.index, orden['q10'], orden['q90'], color='steelblue', alpha=0.15,
                label='intervalo 80%')
ax.axhline(pol['T_crit'], color='crimson', ls='--', lw=1.2, label=f"T crítico = {pol['T_crit']}")
ax.set_xlabel('motores ordenados por RUL real')
ax.set_ylabel('RUL (clip 125)')
ax.set_title('Fase 3 — predicción e intervalo del 80 % (última ventana por motor)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

reglas = pd.DataFrame(pol['reglas']).T.reset_index().rename(columns={'index': 'Regla'})
print('=== comparación de reglas de alarma (test, 100 motores) ===')
display(reglas[['Regla', 'alarmas', 'recall', 'precision', 'coste']].sort_values('coste'))
print('regla elegida:', pol['regla_elegida'])


### 9d. Fase 4 — transferencia a FD002 (6 condiciones) y FD004 (6 condiciones, 2 modos de fallo)

Mismo pipeline, recalibrando el umbral por coste en cada dataset. Se comparan solo los
modelos tabulares: la LSTM se reentrena únicamente en FD001 y FD002 (coste de cómputo).


In [ ]:
cols_trans = ['Dataset', 'Modelo', 'RMSE val/OOF', 'RMSE test', 'MAE test', 'T*',
              'Recall @T*', 'Prec @T*', 'Alarmas']
trans = tab_v3[tab_v3['Features'] == 'health'][cols_trans].copy()
print('=== FASE 4: transferencia del pipeline ===')
display(trans.sort_values(['Dataset', 'RMSE test']).reset_index(drop=True))

piv = tab_v3[tab_v3['Features'] == 'health'].pivot_table(
    index='Modelo', columns='Dataset', values='RMSE test')
print('\nRMSE test por dataset (features de salud):')
display(piv.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
piv.T.plot(kind='bar', ax=ax)
ax.set_ylabel('RMSE test')
ax.set_title('Transferencia: RMSE test por dataset y modelo')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


### 9e. Fase 5 — sistema: inferencia, política y drift

`scripts/predict.py` carga `models/v3_bundle.joblib` (modelo + quantiles + features + versión)
y `models/alarm_policy.json`, y devuelve RUL, intervalo, alarma y **coste esperado de la decisión**.
Para drift: PSI **a nivel de flota** (comparar un solo motor contra la flota mide heterogeneidad,
no drift) y, por motor, % de valores fuera del rango [p01, p99] de entrenamiento.


In [ ]:
import predict
importlib.reload(predict)

_, te5, _ = v3_core.load_fd('FD001')
for eid in (1, 3, 20):
    g = te5[te5['engine_id'] == eid]
    r = predict.predict(g)
    print(f'motor {eid:3d} -> RUL {r["rul"]:6.1f} [{r["q10"]:5.1f}, {r["q90"]:5.1f}]  '
          f'{r["alerta"]:8s} coste_esperado={r["coste_esperado"]:.2f}')

inf, met = predict.report('FD001')
print('\ninforme batch (100 motores):', met)
display(inf.head(5))

dfleet = predict.drift_batch(te5)
print(f"\ndrift de flota -> PSI max {dfleet['psi_max']:.3f} en {dfleet['peor_sensor']} "
      f"(alerta={dfleet['alerta_drift']}, n={dfleet['n_filas']})")
o = predict.out_of_range(te5[te5['engine_id'] == 66])
print(f"motor 66 fuera de rango: {o['peor_pct']:.1f}% en {o['peor_sensor']} (alerta={o['alerta']})")


### 9f. Limitaciones de la v3 (dichas antes de que las pregunte nadie)

- **La `q90 < 30` no funciona**: el intervalo del 80 % mide ~65 ciclos de ancho, así que casi ningún
  motor queda por debajo del umbral. La incertidumbre sirve para priorizar, no para disparar alarmas sola.
- **El umbral se mueve con el dataset**: T* = 22 en FD001, 42.5 en FD002, 45 en FD004. No es transportable
  sin recalibrar (y recalibrar exige fallos reales).
- **OOF mide ventanas, test mide motores**: siguen siendo unidades distintas; el OOF es más pesimista.
- **FD002/FD004 con 1 seed y 3 folds**: sin barra de error. Sirven para ver si transfiere, no para decidir.
- **La LSTM** solo se reentrena en FD001/FD002; en FD004 no se ejecutó por coste (~20 min).
- Un **split temporal** de la RNN se probó y se descartó: validaba bien (14.5) pero en test daba 63,
  porque el tramo final de cada motor tiene un rango de RUL estrecho y el modelo no aprende a distinguir
  motores sanos. El holdout por motor sí cubre todo el rango.
- **El coste 20:1 es una suposición**, no un dato de planta. Cambiarlo mueve T* y el recall.


## 8. Conclusiones (v2 — con los números de esta ejecución)

### Tabla final

| Modelo | RMSE train | RMSE test | MAE test | Tiempo (s) | n motores |
|---|---|---|---|---|---|
| XGBoost baseline (por ciclo, media de ventana) | 14.02 | 26.16 | 20.27 | — | 100 |
| Random Forest (ventana aplanada) | 10.73 | 16.53 | 12.17 | 111.4 | 100 |
| **LightGBM (ventana aplanada)** | **7.34** | **13.42** | **10.33** | **7.7** | **100** |
| Linear Regression (ventana aplanada) | 16.61 | 15.94 | 13.07 | 1.9 | 100 |
| LSTM 2 capas 100-50 (baseline, w=30) | 11.24 | 15.14 | 11.77 | — | 100 |
| LSTM mejor variante (C, w=50) | 12.95 | 15.57 | 10.97 | 213.6 | 93 |


Ganador: **LightGBM** con RMSE test 13.42. Orden completo: LightGBM (13.42) < LSTM 2 capas (15.14) < LSTM mejor variante (15.57) < Linear Regression (15.94) < Random Forest (16.53) < XGBoost baseline (26.16).

### Variantes LSTM

| Variante | RMSE train | RMSE test (sus motores) | RMSE test (mismo n=93) | MAE test | n |
|---|---|---|---|---|---|
| A: LSTM 3 capas 100-64-32 + Dropout 0.2 (w=30) | 10.38 | 15.91 | 16.24 | 11.27 | 100 |
| B: Bidirectional LSTM 100-50 + Dropout 0.2 (w=30) | 9.77 | 15.47 | 15.69 | 10.89 | 100 |
| C: LSTM 2 capas 100-50 + Dropout 0.2 (w=50) | 12.95 | 15.57 | 15.57 | 10.97 | 93 |

La variante elegida es la **C: LSTM 2 capas 100-50 + Dropout 0.2 (w=50)** (RMSE test 15.57 sobre 93 motores,
entrenada 20 epochs). No baja de 15: el objetivo RMSE < 15 **no se cumple** en ninguna variante.
La ventana de 50 ciclos solo existe para 93 de 100 motores, por eso la comparación se hace
también a igual `n`; aun así no mejora al baseline de 2 capas con w=30 (15.14).

### 1. ¿Por qué el LSTM gana al XGBoost aquí?

Al baseline **sí** le gana, y con margen: 26.16 → 15.14 de RMSE.
Pero la razón no es "la recurrencia" sino **qué información ve el modelo**:

- El XGBoost baseline predice **ciclo a ciclo** y en test se promedia la ventana: ve 24 valores sueltos,
  sin saber si el motor está al principio o al final de su vida. Su error se dispara en los motores
  sanos (predice RUL bajo donde debería predecir 125): de ahí que su semáforo no marque **ni un ROJO**
  y falle el 24 % de las zonas.
- Cuando un GBDT ve **la misma ventana aplanada** (720 features), el problema se estrecha: LightGBM
  baja a 13.42 (mejor que cualquier LSTM), Random Forest se queda en
  16.53 (peor, mismo orden) y Linear Regression en 15.94.

Conclusión honesta: **la ventaja sobre el baseline viene de la ventana, no de la arquitectura**.
En FD001 (1 condición, 1 modo de fallo, 100 motores, 20 k ciclos) la señal está en la *forma* de la
ventana y LightGBM la explota mejor que el LSTM, con 7.7 s de entrenamiento frente a
213-361 s por variante LSTM.

### 2. Por qué en AI4I el LSTM perdía

AI4I no es una trayectoria monótona hacia el fallo: es un problema con `type` categórico,
temperaturas, par y desgaste, con target binario y ruido de medición. Allí:

- las features tabulares ya separan bien sin explotar orden temporal;
- el GBDT maneja categóricas e interacciones sin esfuerzo, el LSTM necesita codificarlas y secuenciarlas;
- con pocas muestras por unidad, la recurrencia solo añade varianza.

Resultado: en AI4I el LSTM no aportaba y el GBDT ganaba; aquí pasa algo parecido en cuanto el GBDT
recibe la ventana. **El patrón se repite: la secuencia ayuda cuando aporta información que el modelo
tabular no tiene; si el tabular ya la tiene, el LSTM solo cuesta.**

### 3. Qué modelo usaría en producción

Para este FD001, con estos números: **LightGBM** (RMSE 13.42, MAE 10.33,
7.7 s de entrenamiento, inferencia en milisegundos) y el semáforo encima. Razones:

- mismo rendimiento o mejor que el LSTM, sin GPU ni secuencias servidas;
- reentrenar y versionar es trivial frente a un `.h5` (y aquí `lstm_FD001_best.h5` pesa por arquitectura);
- el semáforo acierta 89.0 % de las zonas críticas con LightGBM.

LSTM solo si el problema real exige modelar dependencias largas que la ventana aplanada no captura
o si ya hay infraestructura de serving para modelos secuenciales.

### 4. Semáforo (RUL < 15 ROJO, < 30 AMARILLO, resto VERDE)

Real: **9 ROJO / 16 AMARILLO / 75 VERDE** (n=100).

| Modelo | ROJO | AMARILLO | VERDE | Zona correcta |
|---|---|---|---|---|
| XGBoost | 0 | 4 | 96 | 76/100 (76.0%) |
| Random Forest | 7 | 11 | 82 | 85/100 (85.0%) |
| LightGBM | 10 | 11 | 79 | 89/100 (89.0%) |
| Linear Regression | 8 | 6 | 86 | 87/100 (87.0%) |
| LSTM 2 capas | 7 | 12 | 81 | 90/100 (90.0%) |
| LSTM best (50 ciclos) | 11 | 15 | 67 | 84/93 (90.3%) |

Ojo con la métrica: acertar zona no es acertar RUL. El LSTM acierta más zona pero **sobreestima**
en los motores verdes; LightGBM es el que más ROJOS marca de verdad a tiempo
(10 detectados frente a 9 reales).

### 5. Limitaciones de esta medición

- Test evaluado solo en la **última ventana** de cada motor (n=100); la variante w=50 solo cubre 93.
- Seed única (42): las diferencias de 1-2 puntos de RMSE entre LSTM/RF/LR están dentro del ruido.
- RUL recortado a 125: los motores sanos nunca se evalúan en su RUL real (>125).
- TensorFlow no es determinista bit a bit; repetir la ejecución mueve las cifras de las variantes.
- El CSV por motor está en `models/test_eval_ultima_ventana.csv` (pred_xgb, pred_lstm, pred_rf,
  pred_lgb, pred_lr, pred_lstm_best) y las métricas crudas en `models/v2_metrics.json`.
